# Bretton et al. (2024): suppression in WM and long-term memory traces -- representational fidelity post hoc

Study-specific analyses of Bretton, Kim, Banich & Lewis-Peacock (2024), *J Cogn Neurosci* 36:2117, run on the core pipeline's output. The core pipeline (`workflows/run_pipeline.py --config examples/config-bretton2024-jocn.example.json`) builds item- and category-level feature-weighted templates from the pre-manipulation memory test and scores the manipulation-phase and post-manipulation-test patterns against them, in VVS, PFC, parietal cortex and hippocampus. This notebook adds what is specific to this design:

1. **Memory flag**: each post-test item is *remembered* (a "sure old" response) or *forgotten* (any other response to an old item) -- derived here from the raw response columns.
2. **Operation**: the WM operation each item received (maintain / replace / suppress) is joined onto its post-test trial from the manipulation phase.
3. **Template check** (manipulation phase): target > other items.
4. **Fidelity by memory** per operation, ROI and level: paired t-tests on subject means (subjects with both outcomes) with subject-bootstrap CIs, and mixed models over all trials, `fidelity ~ memory + (1 + memory | subject)`.

Formal definitions: `docs/lab_methods.md` (section B).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# run from the repository root, so the relative paths in the configs resolve as in the pipeline
REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "_interactive_notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, REPO)

import _interactive_notebooks.rsa_posthoc_common as common
from _interactive_notebooks.rsa_posthoc_common import (
    add_previous_trial_columns, baseline_correct, bin_means, compare_groups_by_bin, difference_scores,
    first_significant_bin, join_item_attributes, load_config, load_master, load_scores, make_bins,
    make_toy_scores, mixed_model, one_sample_by_bin, paired_contrast, pairwise_tests, plot_points,
    plot_timecourse, rm_anova, save_outputs, subject_means, subtract_reference, summarize)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Configuration -- edit these

In [ ]:
CONFIG_PATH = os.environ.get("RSA_CONFIG", "examples/config-bretton2024-jocn.example.json")
ROIS = None                     # None: every ROI in template_rsa.rois (VVS, PFC, parietal, hippocampus)
LEVELS = ["item", "category"]   # feature-weighting levels to analyze
POST_SET = "post_test"          # target-set names in template_rsa.targets
MANIPULATION_SET = "manipulation"

# -- design columns (events.tsv) --
ITEM_COL = "stimulus"
CONDITION_COL = "condition"                                            # the WM operation
OPERATION_SOURCE = {"column": "task", "match": "exact", "value": "study"}  # events that carry each item's operation
RESPONSE_COL, OLD_RESPONSE = "response", "old"                         # post-test old/new judgement
CONFIDENCE_COL, SURE = "confidence", "sure"                            # post-test confidence
OPERATION_ORDER = ["maintain", "replace", "suppress"]

# -- statistics --
ALPHA = 0.05
FDR_METHOD = "fdr_bh"
N_BOOTSTRAP = 10000
SEED = 0

OUTPUT_DIR = os.environ.get("RSA_POSTHOC_OUT")  # None: <output_dir>/posthoc/bretton2024
common.configure(globals())

## Load, flag memory, join the operation

Only old items have templates, so every scored post-test trial is an old, manipulated scene. *Remembered* = answered "old" with "sure" confidence (the paper's hit criterion); everything else = *forgotten*.

In [ ]:
def memory_flag(df):
    hit = (df[RESPONSE_COL] == OLD_RESPONSE) & (df[CONFIDENCE_COL] == SURE)
    return np.where(hit, "remembered", "forgotten")


cfg = load_config(CONFIG_PATH)
OUTPUT_DIR = OUTPUT_DIR or os.path.join(cfg["pipeline"]["output_dir"], "posthoc", "bretton2024")
rois = ROIS or cfg["template_rsa"]["rois"] or list(cfg["rois"])
master = load_master(cfg)

post = pd.concat([load_scores(cfg, roi, POST_SET) for roi in rois], ignore_index=True)
post = post[post["level"].isin(LEVELS) & (post["comparison"] == "target")].copy()
post["memory"] = memory_flag(post)
post = join_item_attributes(post, master, OPERATION_SOURCE, ITEM_COL, [CONDITION_COL])
missing = post[CONDITION_COL].isna().sum()
if missing:
    print(f"(!) {missing} post-test rows have no operation (item not found in OPERATION_SOURCE); dropped")
post = post.dropna(subset=[CONDITION_COL])
present = set(post[CONDITION_COL])
operations = [o for o in OPERATION_ORDER if o in present] + sorted(present - set(OPERATION_ORDER))

counts = (post[post["level"] == LEVELS[0]].drop_duplicates(["roi", "subject", ITEM_COL])
          .query("roi == @rois[0]").groupby(["subject", CONDITION_COL, "memory"]).size().unstack(fill_value=0))
print(f"{post['subject'].nunique()} subjects, ROIs {rois}, levels {LEVELS}; items per subject x operation x memory:")
display(counts)
figures, tables = {}, {"item_counts": counts.reset_index()}

## 1. Template check on the manipulation phase

The paper verified that the weighted templates pick out the item being viewed during the manipulation phase: similarity to its own template (`target`) above similarity to the other items' templates (`related` = the other scenes).

In [ ]:
man = pd.concat([load_scores(cfg, roi, MANIPULATION_SET) for roi in rois], ignore_index=True)
man = man[man["level"].isin(LEVELS)]
man_sub = subject_means(man, ["roi", "level", "comparison"])
template_check = paired_contrast(man_sub, "comparison", "target", "related", by=["roi", "level"])
display(template_check)
tables["template_check_manipulation"] = template_check

## 2. Fidelity by memory outcome, per operation (Fig. 6)

Subject means of post-test fidelity per ROI x level x operation x memory. Paired t-test remembered vs. forgotten within each operation, on the subjects who have both outcomes for that operation (the paper's n = 18), with a subject-bootstrap 95% CI of the difference (`N_BOOTSTRAP` resamples).

In [ ]:
post_sub = subject_means(post, ["roi", "level", CONDITION_COL, "memory"])
memory_tests = paired_contrast(post_sub, "memory", "remembered", "forgotten", by=["roi", "level", CONDITION_COL])

fig, axes = plt.subplots(1, len(LEVELS), figsize=(5.5 * len(LEVELS), 4), sharey=True, squeeze=False)
for ax, level in zip(axes.flat, LEVELS):
    part = post_sub[(post_sub["roi"] == rois[0]) & (post_sub["level"] == level)]
    plot_points(ax, part, CONDITION_COL, "memory", f"{rois[0]}: {level}-level fidelity",
                order=operations, hue_order=["remembered", "forgotten"])
fig.tight_layout()
figures["2_fidelity_by_memory"] = fig
plt.show()
display(memory_tests)
tables["fidelity_remembered_vs_forgotten"] = memory_tests

## 3. Mixed models over all trials

Per ROI x level x operation: `z ~ C(memory, Treatment('remembered')) + (1 + memory | subject)` (statsmodels `mixedlm`), so the memory term is forgotten − remembered. Uses every trial and every subject, including those with only one outcome. Fits that fail (e.g. singular random effects) are listed rather than raised.

In [ ]:
formula = "z ~ C(memory, Treatment('remembered'))"
mixed = mixed_model(post, formula, re_formula="~C(memory, Treatment('remembered'))", by=["roi", "level", CONDITION_COL])
memory_terms = mixed[mixed["term"].str.contains("memory|fit failed", na=False)]
display(memory_terms.drop(columns=["formula", "re_formula"], errors="ignore"))
tables["fidelity_mixed_models"] = mixed

## 4. ROI specificity

The memory difference (remembered − forgotten) per operation in every ROI, side by side: the paper found the suppress-forgotten loss of item-level fidelity in VVS only.

In [ ]:
specificity = memory_tests.pivot_table(index=["level", CONDITION_COL], columns="roi",
                                       values=["mean_diff", "p", "boot_ci_low", "boot_ci_high"]).round(4)
display(specificity)
tables["roi_specificity"] = specificity.reset_index()
tables["roi_specificity"].columns = [" ".join(map(str, c)).strip() for c in tables["roi_specificity"].columns]

## Save outputs

In [ ]:
run_settings = {k: globals()[k] for k in (
    "CONFIG_PATH", "ROIS", "LEVELS", "POST_SET", "MANIPULATION_SET", "ITEM_COL", "CONDITION_COL", "OPERATION_SOURCE",
    "RESPONSE_COL", "OLD_RESPONSE", "CONFIDENCE_COL", "SURE", "ALPHA", "FDR_METHOD", "N_BOOTSTRAP", "SEED")}
for path in save_outputs(OUTPUT_DIR, figures, tables, run_settings):
    print(path)

## Methods

**Scores (core pipeline).** Templates: each manipulated scene's pre-test pattern (2 s presentation, shifted 5 s), in its category's selective voxels (scene > face t-contrast, p < .05, ≥ 10-voxel clusters), weighted by its item-vs-other-89-items t-map (least-squares-all; item level) or by the scene > face t-map (category level). The post-test pattern of the same scene, weighted the same way, is correlated with the template (Pearson r, Fisher z): its representational fidelity.

**Memory and operation.** Remembered = "sure old" on the post-test; forgotten = any other response to an old item. The operation is the one the item received in the manipulation phase.

**Statistics.** Subject means per operation x memory; paired t-tests (subjects with both outcomes) with subject-bootstrap 95% CIs; linear mixed models over trials with random intercepts and memory slopes by subject; repeated for each ROI and weighting level.

**Differences from the paper.** The confound set used to clean the data and the exact VVS label list are assumptions (see `docs/lab_methods.md`); bootstrap CIs resample participants' difference scores rather than refitting pooled trial-level regressions.

## Worked example (toy data)

20 subjects, 30 items each, three operations; forgotten suppress items lose 0.1 Fisher z of fidelity, nothing else differs. The paired contrast should find the suppress effect only.

In [ ]:
rng = np.random.default_rng(1)
toy = pd.DataFrame([{"subject": s, CONDITION_COL: op, "memory": mem,
                     "z": 0.3 + rng.normal(0, 0.15) - (0.1 if (op == "suppress" and mem == "forgotten") else 0)}
                    for s in range(20) for op in ("maintain", "replace", "suppress")
                    for mem in rng.choice(["remembered", "forgotten"], 30, p=[0.75, 0.25])])
toy_tests = paired_contrast(subject_means(toy, [CONDITION_COL, "memory"]), "memory", "remembered", "forgotten",
                            by=[CONDITION_COL])
display(toy_tests[[CONDITION_COL, "n", "mean_diff", "t", "p", "boot_ci_low", "boot_ci_high"]])